# 12 · Corrección de sesgo de selección (Heckman de dos etapas)

La brecha se estima sobre quienes trabajan, y la participación laboral difiere fuertemente por sexo. Si las mujeres que participan están seleccionadas —positiva o negativamente— en atributos no observados correlacionados con el salario, la brecha entre ocupados puede diferir del diferencial poblacional. Los notebooks previos lo declaraban como limitación; aquí lo **cuantificamos** con el procedimiento de dos etapas de Heckman (1979).

**Etapa 1** — probit de participación en la muestra salarial sobre la población en edad de trabajar (25-64 años), con **restricciones de exclusión** que afectan la participación pero, plausiblemente, no el salario condicional: presencia de hijos, estado civil e ingreso no laboral per cápita del hogar. **Etapa 2** — ecuación de salarios sobre los ocupados, agregando la razón inversa de Mills (IMR) como control. Un coeficiente de IMR significativo indica selección; el coeficiente de `mujer` corregido es la brecha neta de selección.

In [1]:
import os
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm
from scipy import stats
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

os.makedirs('outputs/data', exist_ok=True)
RUTA_CASEN = '../../CASEN'
COLS = ['sexo','edad','e6a','o10','oficio4_08','ytrabajocor','expr','activ','varunit',
        's5','ecivil','ytotcorh','numper']
frames = []
for anio in [2022, 2024]:
    df = pd.read_stata(f'{RUTA_CASEN}/casen_{anio}.dta', columns=COLS, convert_categoricals=False)
    df['anio'] = anio
    frames.append(df)
panel = pd.concat(frames, ignore_index=True)

# Poblacion en edad de trabajar 25-64 (margen de participacion mas limpio)
pet = panel[(panel['edad']>=25) & (panel['edad']<=64)].copy()
NIVEL_EDUC = {1:'Básica',2:'Básica',3:'Básica',4:'Básica',5:'Básica',6:'Básica',7:'Básica',
    8:'Media',9:'Media',10:'Media',11:'Media',12:'Técnica sup.',13:'Universitaria',14:'Posgrado',15:'Posgrado'}
pet['mujer'] = (pet['sexo']==2).astype(int)
pet['nivel_grp'] = pet['e6a'].map(NIVEL_EDUC)
pet['edad2'] = pet['edad']**2
pet = pet.dropna(subset=['nivel_grp'])
pet['nivel_grp'] = pd.Categorical(pet['nivel_grp'])
pet['anio'] = pd.Categorical(pet['anio'])
pet['sel'] = ((pet['activ']==1) & (pet['ytrabajocor']>0) & (pet['oficio4_08'].notna()) &
              (pet['o10']>0) & (pet['o10']<=112)).astype(int)
pet['tiene_hijos'] = np.where(pet['s5']>=0, (pet['s5']>0).astype(float), np.nan)
ECIV = {1:'Casado/Conv',2:'Casado/Conv',3:'Casado/Conv',4:'Otro',5:'Otro',6:'Otro',7:'Otro',8:'Soltero'}
pet['ecivil_grp'] = pd.Categorical(pet['ecivil'].map(ECIV))
pet['ing_no_lab_pc_k'] = ((pet['ytotcorh'] - pet['ytrabajocor'].fillna(0)).clip(lower=0) / pet['numper']) / 1e5
pet = pet.dropna(subset=['tiene_hijos','ecivil_grp','ing_no_lab_pc_k'])

sel_h = np.average(pet.loc[pet.mujer==0,'sel'], weights=pet.loc[pet.mujer==0,'expr'])*100
sel_m = np.average(pet.loc[pet.mujer==1,'sel'], weights=pet.loc[pet.mujer==1,'expr'])*100
print(f'Población en edad de trabajar (25-64): {len(pet):,}')
print(f'Tasa de participación en la muestra salarial (ponderada) — hombres: {sel_h:.1f}%  mujeres: {sel_m:.1f}%  (brecha: {sel_h-sel_m:.1f} pp)')

Población en edad de trabajar (25-64): 218,737
Tasa de participación en la muestra salarial (ponderada) — hombres: 82.5%  mujeres: 61.6%  (brecha: 20.9 pp)


## 1. Estimación de dos etapas

In [2]:
def heckman(datos):
    f_sel = ('sel ~ mujer + edad + edad2 + C(nivel_grp) + C(anio) + '
             'tiene_hijos + C(ecivil_grp) + ing_no_lab_pc_k + '
             'mujer:tiene_hijos + mujer:C(ecivil_grp)')
    probit = smf.glm(f_sel, data=datos, family=sm.families.Binomial(sm.families.links.Probit()),
                     freq_weights=datos['expr']).fit()
    Zg = probit.predict(datos, which='linear')
    d = datos.copy()
    d['imr'] = stats.norm.pdf(Zg) / stats.norm.cdf(Zg)
    sub = d[d['sel']==1].copy()
    sub['log_ingreso'] = np.log(sub['ytrabajocor'])
    conteo = sub['oficio4_08'].value_counts()
    val = set(conteo[conteo>=30].index)
    sub['oficio4_grp'] = pd.Categorical(sub['oficio4_08'].apply(lambda c: str(int(c)) if c in val else 'otras'))
    f_base = 'log_ingreso ~ mujer + edad + edad2 + C(nivel_grp) + o10 + C(anio) + C(oficio4_grp)'
    sin = smf.wls(f_base, data=sub, weights=sub['expr']).fit()
    con = smf.wls(f_base + ' + imr', data=sub, weights=sub['expr']).fit()
    return probit, sin, con

probit, sin_imr, con_imr = heckman(pet)
b_sin = (np.exp(sin_imr.params['mujer'])-1)*100
b_con = (np.exp(con_imr.params['mujer'])-1)*100
print('Etapa 1 — restricciones de exclusión (probit de participación):')
print(f'  tiene_hijos:        coef={probit.params["tiene_hijos"]:+.3f}  p={probit.pvalues["tiene_hijos"]:.1e}')
print(f'  ingreso no laboral: coef={probit.params["ing_no_lab_pc_k"]:+.3f}  p={probit.pvalues["ing_no_lab_pc_k"]:.1e}')
print(f'  mujer:tiene_hijos:  coef={probit.params["mujer:tiene_hijos"]:+.3f}  p={probit.pvalues["mujer:tiene_hijos"]:.1e}')
print()
print('Etapa 2 — ecuación de salarios (submuestra 25-64, ocupación 4 dígitos):')
print(f'  Brecha SIN corrección de selección: {b_sin:.2f}%')
print(f'  Brecha CON corrección (Heckman):    {b_con:.2f}%')
print(f'  Razón inversa de Mills (IMR):       coef={con_imr.params["imr"]:+.3f}  p={con_imr.pvalues["imr"]:.1e}  (selección presente)')

Etapa 1 — restricciones de exclusión (probit de participación):
  tiene_hijos:        coef=+0.260  p=0.0e+00
  ingreso no laboral: coef=-0.012  p=0.0e+00
  mujer:tiene_hijos:  coef=-0.389  p=0.0e+00

Etapa 2 — ecuación de salarios (submuestra 25-64, ocupación 4 dígitos):
  Brecha SIN corrección de selección: -15.37%
  Brecha CON corrección (Heckman):    -12.23%
  Razón inversa de Mills (IMR):       coef=-0.129  p=1.6e-21  (selección presente)


## 2. Intervalo de confianza por bootstrap

El error estándar de segunda etapa debe corregirse por el regresor generado (IMR estimado en la primera etapa). Reejecutamos las dos etapas completas sobre 120 remuestras con reemplazo para obtener el intervalo de confianza de la brecha corregida.

In [3]:
rng = np.random.default_rng(2026)
boot = []
for b in range(120):
    bs = pet.sample(frac=1, replace=True, random_state=b)
    try:
        _, _, con_b = heckman(bs)
        boot.append((np.exp(con_b.params['mujer'])-1)*100)
    except Exception:
        continue
boot = np.array(boot)
ic_lo, ic_hi = np.percentile(boot, [2.5, 97.5])
print(f'Brecha corregida por selección: {b_con:.2f}%')
print(f'  IC 95% bootstrap ({len(boot)} réplicas): [{ic_lo:.2f}, {ic_hi:.2f}]')
pd.DataFrame({'metrica':['brecha_sin_correccion','brecha_corregida_seleccion','ic95_inf','ic95_sup',
                          'participacion_h','participacion_m','imr_coef','imr_p'],
              'valor':[b_sin, b_con, ic_lo, ic_hi, sel_h, sel_m,
                       con_imr.params['imr'], con_imr.pvalues['imr']]}).to_csv(
    'outputs/data/correccion_seleccion.csv', index=False, encoding='utf-8-sig')

Brecha corregida por selección: -12.23%
  IC 95% bootstrap (120 réplicas): [-13.86, -10.77]


### Interpretación

La selección es **real y grande en el margen de participación**: en la población de 25-64 años trabaja el ~80% de los hombres frente al ~58% de las mujeres, una brecha de participación de más de 20 puntos. Las restricciones de exclusión (hijos, ingreso no laboral, estado civil) son fuertemente predictoras de la participación, y el coeficiente de la razón inversa de Mills es altamente significativo: hay selección no aleatoria hacia el empleo.

Sin embargo, **la brecha salarial es robusta a corregirla**: pasa de -15.4% (sin corrección, submuestra 25-64) a **-12.2% con la corrección de Heckman** (IC 95% por bootstrap reportado arriba), y sigue siendo grande y estadísticamente significativa. El signo del ajuste indica que el diferencial observado entre ocupados **no es un artefacto de selección positiva**: corregir por selección lo reduce en unos 3 puntos, pero no lo elimina ni cambia el diagnóstico central. Esto matiza —con evidencia y no con supuesto— la conjetura habitual de que la selección necesariamente *subestima* la brecha poblacional: en estos datos, la brecha corregida es algo menor que la observada, pero permanece sustancial.

Caveat estándar de identificación: la validez de la corrección descansa en que las restricciones de exclusión afecten la participación pero no el salario condicional; como toda aplicación de Heckman, el resultado es sensible a ese supuesto y debe leerse como robustez, no como identificación causal.